# ICT304 Tutorial 3 — Question 2: Handling Imbalanced Data (Abalone)

**Group 4** · ICT304 AI System Design (TSA 2026)

**Question:** The Abalone dataset has an imbalanced-data problem. Show the steps and code for dealing with it
before building a machine learning model, and compare the model's performance **before and after** the
imbalance is handled.

**Dataset:** UCI Machine Learning Repository — Abalone (Nash et al., 1994).
<https://archive.ics.uci.edu/ml/datasets/abalone>

## Plan (the steps)

1. Load the data and check it (missing values, bad rows).
2. Define the prediction task and **measure how imbalanced** it is.
3. Split into train/test sets **first** (stratified), so the test set stays untouched.
4. Build a **baseline** model with no fix — the "before" result.
5. Apply four fixes, **only to the training data**:
   class weighting, random undersampling, random oversampling and SMOTE.
6. Compare before vs after with metrics that suit imbalanced data
   (recall, precision, F1, balanced accuracy, PR-AUC) — not just accuracy.
7. Confirm the result with 5-fold stratified cross-validation and draw conclusions.

> **AI tool declaration:** This notebook was drafted with help from Claude (Anthropic).
> The prompts used and screenshots are listed in the presentation's AI declaration slide.

## Step 0 — Setup

`imbalanced-learn` (imblearn) provides the resampling methods. The cell below installs it into the
current kernel if it is missing.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("imblearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])

In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, PrecisionRecallDisplay)

from imblearn.pipeline import Pipeline            # a Pipeline that allows resampling steps
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

SEED = 42                       # fixed seed so results can be repeated
np.random.seed(SEED)

# Folders (the notebook lives in Notebook/, data in Datasets/, outputs go to Results/)
BASE = Path.cwd().parent if Path.cwd().name == "Notebook" else Path.cwd()
DATA_DIR, RESULTS_DIR = BASE / "Datasets", BASE / "Results"
RESULTS_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Project folder:", BASE)

## Step 1 — Load and check the data

The file has no header row, so the column names come from the dataset description (`abalone.names`).
If the file is not in `Datasets/`, it is downloaded from UCI.

In [ ]:
COLUMNS = ["Sex", "Length", "Diameter", "Height", "Whole_weight",
           "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"]
data_file = DATA_DIR / "abalone.data"
if not data_file.exists():
    DATA_DIR.mkdir(exist_ok=True)
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data", data_file)

df = pd.read_csv(data_file, header=None, names=COLUMNS)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
# Physically impossible rows: an abalone cannot have a height of 0 mm.
bad = df[df["Height"] == 0]
print(f"Rows with Height = 0: {len(bad)}")
df = df[df["Height"] > 0].reset_index(drop=True)
print("Shape after removing them:", df.shape)

## Step 2 — Define the task and measure the imbalance

The original target is **Rings** (age ≈ rings + 1.5 years). As a 28-class problem it is extremely
imbalanced — several ring values have only 1–2 examples, so a model can never learn them.

We turn it into a clear, practical binary task: **is the abalone old (Rings ≥ 15, i.e. about 16.5+ years)?**
Only about 1 in 11 abalone is "old", so this is a realistic imbalanced classification problem.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
counts = df["Rings"].value_counts().sort_index()
ax.bar(counts.index, counts.values, color=["#c0392b" if r >= 15 else "#7f8c8d" for r in counts.index])
ax.set_xlabel("Rings"); ax.set_ylabel("Number of abalone")
ax.set_title("Rings distribution (red = 'old', Rings ≥ 15)")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "fig1_rings_distribution.png", bbox_inches="tight")
plt.show()
print("Ring values with fewer than 5 examples:", list(counts[counts < 5].index))

In [ ]:
OLD_THRESHOLD = 15
df["Old"] = (df["Rings"] >= OLD_THRESHOLD).astype(int)
X = df.drop(columns=["Rings", "Old"])     # Rings is removed: it IS the answer (would be data leakage)
y = df["Old"]

class_counts = y.value_counts().sort_index()
ratio = class_counts[0] / class_counts[1]
print(class_counts.rename({0: "Not old (0)", 1: "Old (1)"}))
print(f"Minority share: {y.mean():.1%}   Imbalance ratio ≈ 1 : {ratio:.1f}")

fig, ax = plt.subplots(figsize=(4.2, 3.2))
ax.bar(["Not old", "Old"], class_counts.values, color=["#7f8c8d", "#c0392b"])
for i, v in enumerate(class_counts.values):
    ax.text(i, v, f"{v}", ha="center", va="bottom")
ax.set_title("Class balance of the target"); ax.set_ylabel("Count")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "fig2_class_balance.png", bbox_inches="tight")
plt.show()

## Step 3 — Split first (stratified), then do everything else

* **Stratified** split keeps the same ~9 % "old" share in both train and test sets.
* The split happens **before** scaling or resampling. Resampling the whole dataset first would put
  copies (or SMOTE "neighbours") of test rows into training — a form of **data leakage** that makes
  results look better than they really are.
* Preprocessing (one-hot encode `Sex`, standardise numbers) sits inside the pipeline, so it is fitted
  on training data only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"Train: {len(X_train)} rows, old = {y_train.mean():.1%}")
print(f"Test:  {len(X_test)} rows, old = {y_test.mean():.1%}")

numeric_cols = [c for c in X.columns if c != "Sex"]
preprocess = ColumnTransformer([
    ("sex", OneHotEncoder(handle_unknown="ignore"), ["Sex"]),
    ("num", StandardScaler(), numeric_cols),
])

## Step 4 — Metrics that suit imbalanced data

| Metric | Plain-English meaning |
|---|---|
| Accuracy | % of all predictions that are right — **misleading** when one class dominates |
| Recall (old) | Of the truly old abalone, how many did we find? |
| Precision (old) | Of the abalone we called old, how many really were? |
| F1 (old) | Balance of precision and recall |
| Balanced accuracy | Average of the recall of each class |
| PR-AUC | Area under the precision–recall curve; the "no skill" level equals the minority share (~9 %) |

In [ ]:
def evaluate(name, model, X_te=X_test, y_te=y_test):
    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    return {
        "Method": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Balanced acc.": balanced_accuracy_score(y_te, pred),
        "Recall (old)": recall_score(y_te, pred, zero_division=0),
        "Precision (old)": precision_score(y_te, pred, zero_division=0),
        "F1 (old)": f1_score(y_te, pred, zero_division=0),
        "PR-AUC": average_precision_score(y_te, prob),
        "ROC-AUC": roc_auc_score(y_te, prob),
    }

def make_pipeline(model, sampler=None):
    steps = [("prep", preprocess)]
    if sampler is not None:
        steps.append(("resample", sampler))     # runs during fit() only, never on test data
    steps.append(("model", model))
    return Pipeline(steps)

## Step 5 — "Before": baseline models with no fix

* **Dummy** always predicts the majority class ("not old") — shows the accuracy trap.
* **Logistic Regression** and **Random Forest** trained on the data as it is.

In [ ]:
dummy = make_pipeline(DummyClassifier(strategy="most_frequent")).fit(X_train, y_train)

def logreg(**kw):
    return LogisticRegression(max_iter=2000, random_state=SEED, **kw)

def forest(**kw):
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1, **kw)

baseline_lr = make_pipeline(logreg()).fit(X_train, y_train)
baseline_rf = make_pipeline(forest()).fit(X_train, y_train)

before = pd.DataFrame([
    evaluate("Dummy (always 'not old')", dummy),
    evaluate("Logistic Regression — baseline", baseline_lr),
    evaluate("Random Forest — baseline", baseline_rf),
]).set_index("Method").round(3)
before

The dummy model scores about **91 % accuracy while finding zero old abalone**. The baseline models
also have high accuracy but miss many of the old abalone (low recall). That is the imbalance problem.

## Step 6 — "After": four ways to handle the imbalance

| Method | What it does | Trade-off |
|---|---|---|
| **Class weighting** (`class_weight="balanced"`) | Mistakes on the rare class cost more during training; no rows added or removed | Easiest; model must support weights |
| **Random undersampling** | Randomly drops majority rows until classes are equal | Fast, but throws away ~80 % of data |
| **Random oversampling** | Randomly duplicates minority rows | Keeps all data; can overfit to the copies |
| **SMOTE** | Creates *new synthetic* minority rows between a minority row and its nearest minority neighbours | Less overfitting than copying; can create unrealistic points near the class boundary |

All resampling is inside the pipeline, so it only touches the **training** data.

In [ ]:
samplers = {
    "Random undersampling": RandomUnderSampler(random_state=SEED),
    "Random oversampling": RandomOverSampler(random_state=SEED),
    "SMOTE": SMOTE(k_neighbors=5, random_state=SEED),
}

# Show what each resampler does to the training class counts
Xt_train = preprocess.fit(X_train).transform(X_train)
rows = {"Original": np.bincount(y_train)}
for name, s in samplers.items():
    _, y_res = s.fit_resample(Xt_train, y_train)
    rows[name] = np.bincount(y_res)
resample_counts = pd.DataFrame(rows, index=["Not old (0)", "Old (1)"]).T
resample_counts

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.3))
resample_counts.plot.bar(ax=ax, color=["#7f8c8d", "#c0392b"], rot=0)
ax.set_ylabel("Training rows"); ax.set_title("Training-set class counts after each resampling method")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "fig3_resampled_counts.png", bbox_inches="tight")
plt.show()

In [ ]:
models = {"Logistic Regression": logreg, "Random Forest": forest}
fitted, results = {}, []
for mname, make in models.items():
    variants = {f"{mname} — baseline": make_pipeline(make()),
                f"{mname} — class weighting": make_pipeline(make(class_weight="balanced"))}
    for sname, s in samplers.items():
        variants[f"{mname} — {sname.lower()}"] = make_pipeline(make(), s)
    for vname, pipe in variants.items():
        fitted[vname] = pipe.fit(X_train, y_train)
        results.append(evaluate(vname, pipe))

test_results = pd.DataFrame(results).set_index("Method").round(3)
test_results.to_csv(RESULTS_DIR / "q2_test_results.csv")
test_results

## Step 7 — Compare before vs after

### 7a — 5-fold stratified cross-validation (used to pick the best method fairly)

One test split can be lucky or unlucky, and choosing the "best" method by looking at the test set
would itself be a small leak. So each method is scored with 5-fold stratified cross-validation on the
**training set**, and the best method per model is the one with the highest mean CV F1. The resampling still happens **inside each fold** (only on that fold's training part).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"Recall (old)": "recall", "F1 (old)": "f1",
           "Balanced acc.": "balanced_accuracy", "PR-AUC": "average_precision"}
cv_rows, cv_f1 = [], {}
for mname, make in models.items():
    variants = {"baseline": make_pipeline(make()),
                "class weighting": make_pipeline(make(class_weight="balanced"))}
    for sname, s in samplers.items():
        variants[sname.lower()] = make_pipeline(make(), s)
    for vname, pipe in variants.items():
        sc = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        row = {"Method": f"{mname} — {vname}"}
        for k in scoring:
            v = sc[f"test_{k}"]
            row[k] = f"{v.mean():.3f} ± {v.std():.3f}"
        cv_rows.append(row)
        cv_f1[row["Method"]] = sc["test_F1 (old)"].mean()
cv_results = pd.DataFrame(cv_rows).set_index("Method")
cv_results.to_csv(RESULTS_DIR / "q2_cv_results.csv")
best_names = {m: max([k for k in cv_f1 if k.startswith(m) and not k.endswith("baseline")], key=cv_f1.get)
              for m in models}
print("Best method by mean CV F1:", best_names)
cv_results

In [ ]:
show = ["Recall (old)", "Precision (old)", "F1 (old)", "Balanced acc."]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, mname in zip(axes, models):
    sub = test_results[test_results.index.str.startswith(mname)][show]
    sub.index = [i.split(" — ")[1] for i in sub.index]
    sub.plot.bar(ax=ax, rot=20, width=0.8)
    ax.set_title(f"{mname}: test-set results"); ax.set_ylim(0, 1); ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Before (baseline) vs after (imbalance handled)", y=1.02)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "fig4_before_after_metrics.png", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices: baseline vs the best method (chosen by CV F1) for each model
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, mname in enumerate(models):
    best = best_names[mname]
    for col, vname in enumerate([f"{mname} — baseline", best]):
        ConfusionMatrixDisplay.from_estimator(
            fitted[vname], X_test, y_test, display_labels=["Not old", "Old"],
            cmap="Blues", colorbar=False, ax=axes[row, col])
        axes[row, col].set_title(vname.replace(" — ", "\n"), fontsize=10)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "fig5_confusion_matrices.png", bbox_inches="tight")
plt.show()

In [ ]:
# Precision-recall curves (the dashed line is the 'no skill' level = share of old abalone)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, mname in zip(axes, models):
    for vname in [f"{mname} — baseline", best_names[mname]]:
        PrecisionRecallDisplay.from_estimator(fitted[vname], X_test, y_test, ax=ax,
                                              name=vname.split(" — ")[1])
    ax.axhline(y_test.mean(), ls="--", c="grey", lw=1, label="No skill")
    ax.set_title(mname); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "fig6_pr_curves.png", bbox_inches="tight")
plt.show()

## Step 8 — Summary table for the slides

In [ ]:
summary = test_results.loc[[
    "Logistic Regression — baseline", best_names["Logistic Regression"],
    "Random Forest — baseline", best_names["Random Forest"]],
    ["Accuracy", "Recall (old)", "Precision (old)", "F1 (old)", "Balanced acc.", "PR-AUC"]]
summary.to_csv(RESULTS_DIR / "q2_summary_before_after.csv")
for mname in models:
    b = test_results.loc[f"{mname} — baseline"]; a = test_results.loc[best_names[mname]]
    print(f"{mname}: recall {b['Recall (old)']:.2f} → {a['Recall (old)']:.2f}, "
          f"F1 {b['F1 (old)']:.2f} → {a['F1 (old)']:.2f}, "
          f"accuracy {b['Accuracy']:.2f} → {a['Accuracy']:.2f}  ({best_names[mname].split(' — ')[1]})")
summary

## Conclusions

*(Filled in after running — see the cell output above and `Results/q2_summary_before_after.csv`.)*

## References

* Nash, W. J., Sellers, T. L., Talbot, S. R., Cawthorn, A. J., & Ford, W. B. (1994). *Abalone* [Dataset].
  UCI Machine Learning Repository. https://doi.org/10.24432/C55C7W
* Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority
  over-sampling technique. *Journal of Artificial Intelligence Research, 16*, 321–357.
  https://doi.org/10.1613/jair.953
* Lemaître, G., Nogueira, F., & Aridas, C. K. (2017). Imbalanced-learn: A Python toolbox to tackle the
  curse of imbalanced datasets in machine learning. *Journal of Machine Learning Research, 18*(17), 1–5.
* Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning
  Research, 12*, 2825–2830.
* Huyen, C. (2022). *Designing machine learning systems*. O'Reilly Media. (Chapter 4: Training data — class imbalance.)